# Project 01 — Retail Demand Forecasting & Inventory Optimization

**Dataset:** Walmart M5 Forecasting — [Download from Kaggle](https://www.kaggle.com/competitions/m5-forecasting-accuracy/data)

**Setup:** Download the M5 data files and place them in a `data/m5/` folder next to this notebook.

Files needed:
- `sales_train_validation.csv`
- `calendar.csv`
- `sell_prices.csv`

---

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['figure.figsize'] = (14, 5)
matplotlib.rcParams['axes.facecolor'] = '#111'
matplotlib.rcParams['figure.facecolor'] = '#0a0a0a'
matplotlib.rcParams['text.color'] = '#f0ede8'
matplotlib.rcParams['axes.labelcolor'] = '#a09d98'
matplotlib.rcParams['xtick.color'] = '#5a5755'
matplotlib.rcParams['ytick.color'] = '#5a5755'
matplotlib.rcParams['axes.edgecolor'] = '#2a2a2a'
matplotlib.rcParams['grid.color'] = '#1e1e1e'
matplotlib.rcParams['grid.linestyle'] = '--'

print('Libraries loaded.')

## Step 1 — Load and Explore Data

In [ ]:
# Load data — adjust path if needed
try:
    sales = pd.read_csv('data/m5/sales_train_validation.csv')
    calendar = pd.read_csv('data/m5/calendar.csv')
    prices = pd.read_csv('data/m5/sell_prices.csv')
    print(f'Sales shape: {sales.shape}')
    print(f'Calendar shape: {calendar.shape}')
    print(f'Prices shape: {prices.shape}')
except FileNotFoundError:
    print('M5 data not found. Generating synthetic demo data instead...')
    # --- SYNTHETIC DEMO DATA ---
    np.random.seed(42)
    n_days = 365 * 2
    dates = pd.date_range('2022-01-01', periods=n_days, freq='D')
    n_skus = 50
    records = []
    for sku_id in range(n_skus):
        base = np.random.randint(5, 40)
        trend = np.linspace(0, np.random.uniform(-0.2, 0.3), n_days)
        seasonality = 0.3 * base * np.sin(2 * np.pi * np.arange(n_days) / 365)
        weekly = 0.15 * base * np.sin(2 * np.pi * np.arange(n_days) / 7)
        noise = np.random.normal(0, base * 0.2, n_days)
        demand = np.maximum(0, base + base * trend + seasonality + weekly + noise).astype(int)
        df_sku = pd.DataFrame({'date': dates, 'sku_id': f'SKU_{sku_id:03d}', 'demand': demand,
                                'category': np.random.choice(['Food','Household','Personal Care'], p=[0.5,0.3,0.2]),
                                'store': np.random.choice(['Store_A','Store_B','Store_C'])})
        records.append(df_sku)
    sales_long = pd.concat(records, ignore_index=True)
    sales_long['date'] = pd.to_datetime(sales_long['date'])
    print(f'Synthetic dataset shape: {sales_long.shape}')
    print(sales_long.head())

## Step 2 — EDA: Demand Volatility & CV Analysis

In [ ]:
# Using synthetic data for portability
df = sales_long.copy()

# Coefficient of Variation per SKU
cv_df = df.groupby('sku_id')['demand'].agg(['mean','std']).reset_index()
cv_df['cv'] = cv_df['std'] / cv_df['mean']
cv_df = cv_df.sort_values('cv', ascending=False)

# High-risk SKUs: top 10% by CV
threshold = cv_df['cv'].quantile(0.9)
high_risk = cv_df[cv_df['cv'] >= threshold]
print(f'High-risk SKUs (top 10% CV): {len(high_risk)}')
print(f'CV threshold: {threshold:.3f}')

# Plot CV distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(cv_df['cv'], bins=20, color='#c8f060', alpha=0.8, edgecolor='#0a0a0a')
axes[0].axvline(threshold, color='#f07060', linewidth=2, linestyle='--', label=f'90th pct = {threshold:.2f}')
axes[0].set_title('Coefficient of Variation Distribution', color='#f0ede8', fontsize=13)
axes[0].set_xlabel('CV (Std / Mean)')
axes[0].legend()

# Daily demand by category
cat_daily = df.groupby(['date','category'])['demand'].sum().reset_index()
for cat, grp in cat_daily.groupby('category'):
    axes[1].plot(grp['date'], grp['demand'].rolling(14).mean(), label=cat, linewidth=1.5)
axes[1].set_title('14-Day Rolling Demand by Category', color='#f0ede8', fontsize=13)
axes[1].legend()

plt.tight_layout()
plt.savefig('eda_demand_volatility.png', dpi=150, bbox_inches='tight')
plt.show()
print('EDA complete. Chart saved as eda_demand_volatility.png')

## Step 3 — Forecasting: ARIMA Baseline vs. Prophet Ensemble

In [ ]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_percentage_error

# Pick one SKU to demonstrate
sku = df['sku_id'].value_counts().index[0]
sku_df = df[df['sku_id'] == sku].set_index('date')['demand'].sort_index()

train_size = int(len(sku_df) * 0.8)
train, test = sku_df.iloc[:train_size], sku_df.iloc[train_size:]

print(f'Forecasting SKU: {sku}')
print(f'Train size: {len(train)} days | Test size: {len(test)} days')

# ARIMA baseline
model = ARIMA(train, order=(2, 1, 2))
arima_fit = model.fit()
arima_pred = arima_fit.forecast(steps=len(test))
arima_mape = mean_absolute_percentage_error(test, np.maximum(arima_pred, 0)) * 100
print(f'ARIMA MAPE: {arima_mape:.1f}%')

In [ ]:
# Prophet model
try:
    from prophet import Prophet
    prophet_available = True
except ImportError:
    print('Prophet not installed. Run: pip install prophet')
    print('Using moving average as Prophet substitute for demo...')
    prophet_available = False

if prophet_available:
    prophet_df = train.reset_index().rename(columns={'date': 'ds', 'demand': 'y'})
    m = Prophet(yearly_seasonality=True, weekly_seasonality=True, daily_seasonality=False, interval_width=0.9)
    m.fit(prophet_df)
    future = m.make_future_dataframe(periods=len(test))
    forecast = m.predict(future)
    prophet_pred = forecast['yhat'].iloc[-len(test):].values
    prophet_pred = np.maximum(prophet_pred, 0)
    prophet_mape = mean_absolute_percentage_error(test, prophet_pred) * 100
    print(f'Prophet MAPE: {prophet_mape:.1f}%')

    # Ensemble (60% Prophet, 40% ARIMA)
    ensemble_pred = 0.6 * prophet_pred + 0.4 * np.maximum(arima_pred, 0)
    ensemble_mape = mean_absolute_percentage_error(test, ensemble_pred) * 100
    print(f'Ensemble MAPE: {ensemble_mape:.1f}%')
else:
    # Substitute: 14-day rolling mean
    prophet_pred = np.array([train.iloc[-14:].mean()] * len(test))
    prophet_mape = mean_absolute_percentage_error(test, prophet_pred) * 100
    ensemble_pred = 0.6 * prophet_pred + 0.4 * np.maximum(arima_pred, 0)
    ensemble_mape = mean_absolute_percentage_error(test, ensemble_pred) * 100
    print(f'Moving avg MAPE (Prophet substitute): {prophet_mape:.1f}%')
    print(f'Ensemble MAPE: {ensemble_mape:.1f}%')

# Plot
fig, ax = plt.subplots(figsize=(14, 5))
ax.plot(test.index, test.values, label='Actual', color='#f0ede8', linewidth=1.5)
ax.plot(test.index, np.maximum(arima_pred, 0), label=f'ARIMA (MAPE {arima_mape:.1f}%)', color='#60a8f0', linewidth=1.2, linestyle='--')
ax.plot(test.index, ensemble_pred, label=f'Ensemble (MAPE {ensemble_mape:.1f}%)', color='#c8f060', linewidth=2)
ax.set_title(f'Demand Forecast vs Actual — {sku}', color='#f0ede8', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig('forecast_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Forecast chart saved as forecast_comparison.png')

## Step 4 — Inventory Optimization: EOQ + Dynamic Safety Stock

In [ ]:
# Parameters
LEAD_TIME_DAYS = 3
HOLDING_COST_PCT = 0.25   # 25% of unit cost per year
ORDERING_COST = 50        # $ per order
UNIT_COST = 8             # avg unit cost $
SERVICE_LEVEL_Z = 1.65    # 95% service level

# Compute inventory metrics per SKU
inv_results = []
for sku_id, grp in df.groupby('sku_id'):
    avg_demand = grp['demand'].mean()
    std_demand = grp['demand'].std()
    annual_demand = avg_demand * 365

    # Economic Order Quantity
    holding_cost_unit = UNIT_COST * HOLDING_COST_PCT
    eoq = np.sqrt((2 * annual_demand * ORDERING_COST) / holding_cost_unit)

    # Safety stock based on forecast uncertainty
    safety_stock = SERVICE_LEVEL_Z * std_demand * np.sqrt(LEAD_TIME_DAYS)

    # Reorder point
    rop = avg_demand * LEAD_TIME_DAYS + safety_stock

    # Annual carrying cost
    carrying_cost = (eoq / 2 + safety_stock) * UNIT_COST * HOLDING_COST_PCT

    inv_results.append({
        'sku_id': sku_id,
        'avg_daily_demand': round(avg_demand, 1),
        'cv': round(std_demand / avg_demand, 3) if avg_demand > 0 else 0,
        'eoq_units': round(eoq),
        'safety_stock_units': round(safety_stock),
        'reorder_point': round(rop),
        'annual_carrying_cost_usd': round(carrying_cost, 2)
    })

inv_df = pd.DataFrame(inv_results)
total_carrying = inv_df['annual_carrying_cost_usd'].sum()

print(f'Total simulated annual carrying cost: ${total_carrying:,.0f}')
print(f'\nTop 10 highest carrying cost SKUs:')
print(inv_df.nlargest(10, 'annual_carrying_cost_usd')[['sku_id','avg_daily_demand','cv','eoq_units','safety_stock_units','reorder_point','annual_carrying_cost_usd']].to_string(index=False))

In [ ]:
# Carrying cost savings from CV-optimized safety stock
# Baseline: industry uses fixed safety stock = 14 days
inv_df['baseline_ss'] = inv_df['avg_daily_demand'] * 14
inv_df['baseline_cost'] = (inv_df['eoq_units'] / 2 + inv_df['baseline_ss']) * UNIT_COST * HOLDING_COST_PCT
inv_df['optimized_cost'] = inv_df['annual_carrying_cost_usd']
inv_df['savings'] = inv_df['baseline_cost'] - inv_df['optimized_cost']

total_savings = inv_df['savings'].sum()
print(f'Annual carrying cost SAVINGS from dynamic safety stock: ${total_savings:,.0f}')

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

top10 = inv_df.nlargest(10, 'savings')
axes[0].barh(top10['sku_id'], top10['savings'], color='#c8f060', alpha=0.85)
axes[0].set_title('Top 10 SKUs by Carrying Cost Savings', color='#f0ede8', fontsize=12)
axes[0].set_xlabel('Annual Savings ($)')

axes[1].scatter(inv_df['cv'], inv_df['annual_carrying_cost_usd'], alpha=0.6, color='#60a8f0', s=40)
axes[1].set_xlabel('Coefficient of Variation (CV)')
axes[1].set_ylabel('Annual Carrying Cost ($)')
axes[1].set_title('CV vs Carrying Cost per SKU', color='#f0ede8', fontsize=12)

plt.tight_layout()
plt.savefig('inventory_optimization.png', dpi=150, bbox_inches='tight')
plt.show()

# Export for Tableau
inv_df.to_csv('inventory_reorder_signals.csv', index=False)
print('\nReorder signals exported to inventory_reorder_signals.csv')
print('\n=== PROJECT 01 COMPLETE ===')
print(f'ARIMA MAPE: {arima_mape:.1f}% | Ensemble MAPE: {ensemble_mape:.1f}%')
print(f'Simulated annual carrying cost savings: ${total_savings:,.0f}')